# 07a: Generate Knowledge Base for HLC

**Runtime: A100 or G4** (Mistral-7B needs GPU VRAM)

Uses Mistral-7B-Instruct to generate 1000+ diverse, high-quality declarative facts
across 25+ domains. These become the mind's permanent memory columns.

**What this produces:**
- `knowledge_base.json` — 1000+ facts organized by domain
- `knowledge_base.py` — Same facts as a Python list for easy import
- `strategies.json` — 50+ reasoning strategies (STRATEGY: prefix)

**Requirements for each fact:**
- Declarative (not a question)
- Self-contained (one complete thought)
- Atomic (one fact per sentence, max 2 sentences)
- Accurate and specific
- Diverse vocabulary

**Time:** ~30-45 min on A100 | **Cost:** $0

In [ ]:
!pip install -q transformers accelerate
!nvidia-smi | head -4

In [ ]:
import torch
import json
import re
import time
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
OUTPUT_DIR = Path("/content")

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Model loaded.")

In [ ]:
def generate(prompt, max_new_tokens=2000, temperature=0.7):
    """Generate text from Mistral."""
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=1024,
    ).to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    full_text = tokenizer.decode(output[0], skip_special_tokens=True)
    response = full_text[len(tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)):].strip()
    return response


def parse_numbered_list(text):
    """Extract numbered items from LLM output."""
    facts = []
    for line in text.strip().split("\n"):
        line = line.strip()
        if not line:
            continue
        # Remove numbering: "1. ", "1) ", "- ", "* "
        cleaned = re.sub(r"^\d+[\.)\-]\s*", "", line)
        cleaned = re.sub(r"^[-*]\s*", "", cleaned)
        cleaned = cleaned.strip()
        if not cleaned:
            continue
        # Must be a proper sentence (starts upper, ends with period)
        if len(cleaned) < 15:
            continue
        if cleaned.endswith("?"):
            continue
        # Ensure it ends with punctuation
        if not cleaned[-1] in ".!":
            cleaned += "."
        # Ensure starts with capital
        cleaned = cleaned[0].upper() + cleaned[1:]
        facts.append(cleaned)
    return facts


print("Generation functions ready.")

## Step 1: Define domains and generate facts

Each domain gets a carefully crafted prompt asking for 50 diverse, atomic, declarative facts.
We generate in batches of 25 to keep quality high (LLMs degrade on very long lists).

In [ ]:
# 25+ domains covering the breadth of human knowledge
DOMAINS = {
    "physics": "physics and mechanics (forces, motion, waves, thermodynamics, optics, relativity, quantum mechanics)",
    "chemistry": "chemistry (elements, compounds, reactions, organic chemistry, materials, bonding, solutions)",
    "biology": "biology and life sciences (genetics, ecology, evolution, microbiology, anatomy, botany, zoology)",
    "medicine": "medicine and human health (diseases, anatomy, nutrition, immune system, pharmacology, mental health)",
    "astronomy": "astronomy and space science (planets, stars, galaxies, black holes, cosmology, space exploration)",
    "earth_science": "earth science and geology (plate tectonics, volcanoes, minerals, oceans, atmosphere, fossils)",
    "mathematics": "mathematics (algebra, geometry, calculus, statistics, number theory, topology, set theory)",
    "computer_science": "computer science (algorithms, data structures, networking, operating systems, cryptography, databases)",
    "history_ancient": "ancient history (Egypt, Greece, Rome, Mesopotamia, China, India, Mesoamerica, early civilizations)",
    "history_modern": "modern history from 1500 to present (colonialism, revolutions, world wars, cold war, globalization)",
    "geography": "world geography (continents, countries, rivers, mountains, deserts, climate zones, population)",
    "economics": "economics and finance (markets, trade, monetary policy, inflation, GDP, banking, economic theory)",
    "psychology": "psychology (cognition, memory, perception, development, social behavior, disorders, neuroscience)",
    "philosophy": "philosophy (ethics, epistemology, logic, metaphysics, major thinkers, thought experiments)",
    "linguistics": "language and linguistics (grammar, phonetics, language families, writing systems, semantics, translation)",
    "art_architecture": "art and architecture (painting styles, sculpture, famous works, architectural movements, design principles)",
    "music": "music (theory, instruments, genres, composers, acoustics, rhythm, harmony)",
    "literature": "literature (genres, famous works, narrative techniques, poetry, mythology, storytelling traditions)",
    "engineering": "engineering (structural, electrical, mechanical, civil, materials science, manufacturing)",
    "ecology_environment": "ecology and environment (ecosystems, climate change, biodiversity, conservation, pollution, sustainability)",
    "food_agriculture": "food science and agriculture (crops, cooking chemistry, fermentation, nutrition, soil, farming techniques)",
    "animals": "animals and zoology (mammals, birds, marine life, insects, behavior, adaptation, endangered species)",
    "materials_everyday": "everyday materials and how things work (glass, paper, plastic, steel, concrete, textiles, electronics)",
    "ocean_marine": "oceans and marine science (currents, tides, deep sea, coral reefs, marine organisms, salinity)",
    "weather_climate": "weather and climate (storms, precipitation, wind patterns, seasons, atmospheric pressure, climate systems)",
    "sociology": "sociology and human society (culture, institutions, social structures, demographics, urbanization, religion)",
    "law_governance": "law and governance (legal systems, constitutions, human rights, international law, democracy, justice)",
    "transportation": "transportation and vehicles (aviation, railways, ships, automobiles, roads, navigation, logistics)",
    "energy": "energy systems (solar, nuclear, fossil fuels, wind, hydroelectric, batteries, power grids, efficiency)",
    "sports_games": "sports and games (rules, physics of sports, strategy, history of major sports, Olympics, chess)",
}

print(f"Defined {len(DOMAINS)} knowledge domains.")
for name, desc in DOMAINS.items():
    print(f"  {name}: {desc[:60]}...")

In [ ]:
def generate_facts_for_domain(domain_name, domain_description, batch_size=25, num_batches=2):
    """
    Generate facts for one domain in batches.
    Two batches of 25 = 50 facts per domain.
    """
    all_facts = []

    for batch_idx in range(num_batches):
        # Vary the prompt slightly for each batch to get diverse facts
        if batch_idx == 0:
            focus = "Cover fundamental concepts, key principles, and important facts."
        else:
            focus = "Cover advanced topics, surprising facts, practical applications, and lesser-known details. Do NOT repeat any facts from before."

        # Include already-generated facts to avoid duplicates
        already_text = ""
        if all_facts:
            sample = all_facts[:10]  # Show first 10 as context
            already_text = (
                f"\n\nYou already generated these facts (DO NOT repeat them or anything similar):\n"
                + "\n".join(f"- {f}" for f in sample)
            )

        prompt = (
            f"[INST] Generate exactly {batch_size} declarative facts about {domain_description}.\n\n"
            f"STRICT RULES:\n"
            f"- Each fact must be 1-2 sentences, self-contained, and specific\n"
            f"- Each fact must be a DECLARATIVE STATEMENT (never a question)\n"
            f"- Include specific numbers, names, or measurements where possible\n"
            f"- Each fact should teach something distinct — no overlap between facts\n"
            f"- Facts must be accurate and verifiable\n"
            f"- Use clear, precise English\n"
            f"- {focus}\n"
            f"{already_text}\n\n"
            f"Output ONLY the numbered facts, one per line, numbered 1-{batch_size}. [/INST]\n"
        )

        raw = generate(prompt, max_new_tokens=2500, temperature=0.7)
        facts = parse_numbered_list(raw)

        # Quality filter: remove very short, very long, or question-like
        good_facts = []
        for f in facts:
            words = f.split()
            if len(words) < 5 or len(words) > 60:
                continue
            if f.endswith("?"):
                continue
            good_facts.append(f)

        all_facts.extend(good_facts)
        print(f"    Batch {batch_idx + 1}: {len(good_facts)} facts (total: {len(all_facts)})")

    return all_facts


print("Fact generation function ready.")

In [ ]:
# === GENERATE ALL FACTS ===
# This is the main generation loop. ~25-35 minutes on A100.

knowledge_base = {}  # domain -> list of facts
total_facts = 0
start_time = time.time()

for i, (domain_name, domain_desc) in enumerate(DOMAINS.items()):
    print(f"\n[{i+1}/{len(DOMAINS)}] Generating: {domain_name}")
    t0 = time.time()

    facts = generate_facts_for_domain(domain_name, domain_desc)
    knowledge_base[domain_name] = facts
    total_facts += len(facts)

    elapsed = time.time() - t0
    print(f"  -> {len(facts)} facts in {elapsed:.0f}s (running total: {total_facts})")

total_time = time.time() - start_time
print(f"\n{'='*60}")
print(f"GENERATION COMPLETE")
print(f"  Total facts: {total_facts}")
print(f"  Domains: {len(knowledge_base)}")
print(f"  Time: {total_time/60:.1f} minutes")
print(f"  Average per domain: {total_facts/len(knowledge_base):.0f} facts")
print(f"{'='*60}")

In [ ]:
# Preview facts from a few domains
for domain in ["physics", "history_ancient", "animals", "economics", "music"]:
    if domain in knowledge_base:
        print(f"\n=== {domain.upper()} (showing 5/{len(knowledge_base[domain])}) ===")
        for fact in knowledge_base[domain][:5]:
            print(f"  - {fact}")

## Step 2: Generate reasoning strategies

These become STRATEGY columns — meta-knowledge about *how to think*,
not *what to know*. Like cognitive tools.

In [ ]:
STRATEGY_CATEGORIES = [
    "problem-solving and decomposition",
    "logical reasoning and argumentation",
    "scientific thinking and hypothesis testing",
    "creative thinking and analogy",
    "critical thinking and bias detection",
    "mathematical and quantitative reasoning",
    "decision making under uncertainty",
    "learning and knowledge acquisition",
    "communication and explanation",
    "systems thinking and feedback loops",
]

strategies = []

for i, category in enumerate(STRATEGY_CATEGORIES):
    print(f"[{i+1}/{len(STRATEGY_CATEGORIES)}] Strategies for: {category}")

    prompt = (
        f"[INST] Generate exactly 5 reasoning strategies for {category}.\n\n"
        f"Each strategy must be:\n"
        f"- A general-purpose thinking rule (not domain-specific)\n"
        f"- Actionable and concrete\n"
        f"- 1-2 sentences\n"
        f"- Written as an imperative instruction (e.g., 'When X, do Y')\n\n"
        f"Output ONLY the strategies, numbered 1-5. [/INST]\n"
    )

    raw = generate(prompt, max_new_tokens=600, temperature=0.7)
    parsed = parse_numbered_list(raw)

    # Add STRATEGY: prefix
    for s in parsed[:5]:
        # Remove "STRATEGY:" if Mistral already added it
        clean = re.sub(r"^STRATEGY:\s*", "", s, flags=re.IGNORECASE)
        strategies.append(f"STRATEGY: {clean}")

    print(f"  -> {min(len(parsed), 5)} strategies")

print(f"\nTotal strategies: {len(strategies)}")
for s in strategies[:5]:
    print(f"  {s}")

## Step 3: Deduplicate and quality filter

Remove near-duplicates using simple text similarity.
Also remove facts that are too similar to the existing 60 seed facts.

In [ ]:
# Existing seed facts to avoid duplicating
EXISTING_SEED_FACTS = [
    "Water boils at 100 degrees Celsius at standard atmospheric pressure.",
    "Water freezes at 0 degrees Celsius and becomes ice.",
    "Light travels at approximately 300,000 kilometers per second in a vacuum.",
    "Gravity is a force that pulls objects toward each other.",
    "Energy cannot be created or destroyed, only transformed from one form to another.",
    "Heat transfers from hotter objects to cooler objects until thermal equilibrium is reached.",
    "Sound travels through air at approximately 343 meters per second.",
    "Electricity is the flow of electrons through a conductor.",
    "All living organisms are made of cells, which are the basic units of life.",
    "DNA contains the genetic instructions for the development and function of living organisms.",
    "Photosynthesis is the process by which plants convert sunlight, water, and carbon dioxide into glucose and oxygen.",
    "The human brain contains approximately 86 billion neurons.",
    "Atoms are the basic building blocks of matter.",
    "The periodic table organizes elements by their atomic number.",
    "Pi is approximately 3.14159.",
    "The Earth orbits the Sun once every 365.25 days.",
    "Computers process information using binary digits.",
    "Fire requires fuel, heat, and oxygen to burn.",
]


def normalize(text):
    """Normalize text for dedup comparison."""
    return text.lower().strip().rstrip(".")


def is_too_similar(fact, existing_set, threshold=0.8):
    """Check if a fact is too similar to any in the existing set (word overlap)."""
    fact_words = set(normalize(fact).split())
    for existing in existing_set:
        existing_words = set(normalize(existing).split())
        if not fact_words or not existing_words:
            continue
        overlap = len(fact_words & existing_words) / min(len(fact_words), len(existing_words))
        if overlap > threshold:
            return True
    return False


# Deduplicate
all_facts_flat = []
for domain, facts in knowledge_base.items():
    all_facts_flat.extend(facts)

print(f"Before dedup: {len(all_facts_flat)} facts")

# Remove exact duplicates
seen_normalized = set()
unique_facts = []
for fact in all_facts_flat:
    norm = normalize(fact)
    if norm not in seen_normalized:
        seen_normalized.add(norm)
        unique_facts.append(fact)

print(f"After exact dedup: {len(unique_facts)} facts")

# Remove facts too similar to seed facts
filtered_facts = []
seed_removed = 0
for fact in unique_facts:
    if is_too_similar(fact, EXISTING_SEED_FACTS, threshold=0.7):
        seed_removed += 1
    else:
        filtered_facts.append(fact)

print(f"Removed {seed_removed} facts too similar to seed_basic.py")

# Remove near-duplicates within the generated set
final_facts = []
near_dupes = 0
for fact in filtered_facts:
    if not is_too_similar(fact, final_facts, threshold=0.8):
        final_facts.append(fact)
    else:
        near_dupes += 1

print(f"Removed {near_dupes} near-duplicates within generated set")
print(f"\nFinal knowledge base: {len(final_facts)} unique facts")
print(f"Final strategies: {len(strategies)}")
print(f"Combined with seed_basic.py (70): {len(final_facts) + len(strategies) + 70} total columns")

In [ ]:
# Domain distribution of final facts
domain_counts = {}
for domain, facts in knowledge_base.items():
    count = sum(1 for f in facts if f in final_facts)
    domain_counts[domain] = count

print("Domain distribution:")
for domain, count in sorted(domain_counts.items(), key=lambda x: -x[1]):
    bar = '#' * (count // 2)
    print(f"  {domain:25s} {count:3d} {bar}")

## Step 4: Save and download

Two formats:
1. `knowledge_base.json` — Full structured data with domain tags
2. `knowledge_base.py` — Python file for direct import in the experiment

In [ ]:
# === Save as JSON (structured) ===
output_json = {
    "metadata": {
        "total_facts": len(final_facts),
        "total_strategies": len(strategies),
        "domains": len(knowledge_base),
        "generated_by": MODEL_NAME,
        "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    },
    "facts_by_domain": {},
    "strategies": strategies,
    "all_facts": final_facts,
}

# Rebuild domain mapping for final facts
for domain, facts in knowledge_base.items():
    domain_final = [f for f in facts if f in set(final_facts)]
    if domain_final:
        output_json["facts_by_domain"][domain] = domain_final

json_path = OUTPUT_DIR / "knowledge_base.json"
with open(json_path, "w") as f:
    json.dump(output_json, f, indent=2)
print(f"Saved {json_path} ({json_path.stat().st_size / 1024:.0f} KB)")


# === Save as Python file (for direct import) ===
py_path = OUTPUT_DIR / "knowledge_base.py"
with open(py_path, "w") as f:
    f.write('"""Generated knowledge base for HLC — 1000+ diverse facts across 25+ domains."""\n\n')
    f.write(f'# Generated by {MODEL_NAME} on {time.strftime("%Y-%m-%d")}\n')
    f.write(f'# Total: {len(final_facts)} facts + {len(strategies)} strategies\n\n')
    f.write('GENERATED_FACTS = [\n')
    for fact in final_facts:
        escaped = fact.replace('\\', '\\\\').replace('"', '\\"')
        f.write(f'    "{escaped}",\n')
    f.write(']\n\n')
    f.write('GENERATED_STRATEGIES = [\n')
    for s in strategies:
        escaped = s.replace('\\', '\\\\').replace('"', '\\"')
        f.write(f'    "{escaped}",\n')
    f.write(']\n')

print(f"Saved {py_path} ({py_path.stat().st_size / 1024:.0f} KB)")

In [ ]:
# === Final summary ===
print("=" * 60)
print("KNOWLEDGE BASE GENERATION COMPLETE")
print("=" * 60)
print(f"")
print(f"  Knowledge facts:  {len(final_facts)}")
print(f"  Strategies:       {len(strategies)}")
print(f"  Domains covered:  {len(knowledge_base)}")
print(f"")
print(f"  + seed_basic.py:  60 facts + 10 strategies")
print(f"  ─────────────────────────────────────────")
print(f"  TOTAL COLUMNS:    {len(final_facts) + len(strategies) + 70}")
print(f"")
print(f"Files to download:")
print(f"  1. knowledge_base.json  (structured, with domain tags)")
print(f"  2. knowledge_base.py    (Python import for experiment)")
print(f"")
print(f"Place both in: experiments/ directory")
print("=" * 60)

In [ ]:
# Download both files
from google.colab import files

files.download(str(json_path))
files.download(str(py_path))